# Kite gen+cost — East Coast, 2011–2020, depths 50/100/200 m

Reads the gridded `EastCoast_Current_{year}.npz` decade files (never loading a 23 GB
array whole — streams time-blocks), evaluates each kite design across all years at the
three operating depths, and writes one file per design named
`kite_{kW}kW_{rfsmin}-{rfsmax}mRFS`. Output carries a depth axis: `Energy_pu (time, 3, sites)`.

Two passes: (1) extract the 3 depth layers for valid ocean sites to small per-year temps;
(2) apply each design's power surface. The current files are read once regardless of
how many designs you run.

In [1]:
# ================= CONFIG =================
import os, glob, zipfile, time
import numpy as np, pandas as pd
from numpy.lib import format as npformat

BASE = r"C:\Users\rmiller9\Documents\East Coast Model"
CUR_DIR   = BASE + r"\Resource Data\Current\EastCoast"
SURF_DIR  = BASE + r"\Tech Designs\Current\Power Surfaces"
OUT_DIR   = BASE + r"\Tech Outputs\Current"
TMP_DIR   = CUR_DIR + r"\_kite_tmp"

YEARS = list(range(2011, 2021))
DEVICE_IDS = list(range(10))          # kite_0 .. kite_9
OPERATING_DEPTHS = [50, 100, 150, 200]     # m — must exist in the current depth axis
FCR = 0.10
RES_DEG, RES_KM = 0.08, 8
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(TMP_DIR, exist_ok=True)

def cur_file(y): return os.path.join(CUR_DIR, f"EastCoast_Current_{y}.npz")
for y in YEARS: assert os.path.exists(cur_file(y)), f"missing {cur_file(y)}"

# depth-axis indices for the operating depths
depth_axis = np.load(cur_file(YEARS[0]), allow_pickle=True)["depth"]
DEPTH_IDX = [int(np.where(depth_axis == D)[0][0]) for D in OPERATING_DEPTHS]
print("operating depths", OPERATING_DEPTHS, "-> current indices", DEPTH_IDX)

operating depths [50, 100, 150, 200] -> current indices [2, 4, 5, 6]


## Helpers — 4D time-block streaming, RFS/name derivation

In [ ]:
def _hdr(fp):
    v=npformat.read_magic(fp)
    try: return npformat._read_array_header(fp,v)
    except AttributeError:
        return npformat.read_array_header_1_0(fp) if v==(1,0) else npformat.read_array_header_2_0(fp)

def stream_current(path, depth_idx, keep_cols=None, block=200):
    """Yield (r0, r1, (nb, n_depth, n_cols)) from ocean_speed without loading it whole."""
    with zipfile.ZipFile(path) as zf, zf.open("ocean_speed.npy") as f:
        shape,fort,dt=_hdr(f); T,ND,NLA,NLO=shape; per_t=ND*NLA*NLO; it=dt.itemsize
        ncell=NLA*NLO
        for r0 in range(0,T,block):
            r1=min(r0+block,T); nb=r1-r0
            arr=np.frombuffer(f.read(nb*per_t*it),dt).reshape(nb,ND,NLA,NLO)[:,depth_idx,:,:].reshape(nb,len(depth_idx),ncell)
            yield r0, r1, (arr[:,:,keep_cols] if keep_cols is not None else arr)

def n_time(path):
    with zipfile.ZipFile(path) as zf, zf.open("ocean_speed.npy") as f:
        return _hdr(f)[0][0]

def _vfmt(v):
    s=f"{v:g}"; return s if "." in s else s+".0"

def load_design(did):
    dv=pd.read_csv(os.path.join(SURF_DIR,f"kite_{did}Depth_Vector.csv"))["Depth"].values.astype(float)
    vv=pd.read_csv(os.path.join(SURF_DIR,f"kite_{did}Velocity_Vector.csv"))["Velocity"].values.astype(float)
    pm=pd.read_csv(os.path.join(SURF_DIR,f"kite_{did}.csv"),header=None).values.astype(float)  # (depth,vel)
    capex,opex,rated=pd.read_csv(os.path.join(SURF_DIR,f"kite{did}costInp.csv"),header=None).values.flatten()[:3]
    # RFS range = min..max plateau velocity across depth rows
    plateau=[vv[np.argmax(pm[i]>=0.99*pm.max())] for i in range(len(dv))]
    lo,hi=min(plateau),max(plateau)
    rfs = _vfmt(lo) if lo==hi else f"{_vfmt(lo)}-{_vfmt(hi)}"
    name=f"kite_{round(float(rated))}kW_{rfs}mRFS"
    # 1D power curve at each operating depth (50/100/200 are exact grid rows)
    rows=[pm[int(np.where(dv==D)[0][0])] for D in OPERATING_DEPTHS]
    return dict(name=name, vv=vv, rows=rows, capex=float(capex), opex=float(opex), rated=float(rated))

## Pass 1 — extract 3 depths for valid ocean sites (once)

In [ ]:
# valid mask + grid from year 1
D0 = np.load(cur_file(YEARS[0]), allow_pickle=True)
lat, lon = D0["lat"], D0["lon"]; D0.close()
ncell = len(lat)*len(lon)
lon_m, lat_m = np.meshgrid(lon, lat)
LatLong_all = np.column_stack([lat_m.ravel(), lon_m.ravel()]).astype(np.float32)

print("scanning year 1 for valid ocean sites...")
valid = np.zeros((len(OPERATING_DEPTHS), ncell), bool)
for r0,r1,a in stream_current(cur_file(YEARS[0]), DEPTH_IDX):
    valid |= np.any(~np.isnan(a), axis=0)
keep_cols = np.flatnonzero(valid.any(axis=0))
ValidSites = valid[:, keep_cols]
LatLong = LatLong_all[keep_cols]
n_keep = keep_cols.size
print(f"grid {ncell:,} cells -> {n_keep:,} valid ocean sites")

# extract each year's 3-depth speed for kept sites -> small temp (NaN->0)
year_len=[]; datetimes=[]
for y in YEARS:
    tp = os.path.join(TMP_DIR, f"_curr_{y}.npy")
    T = n_time(cur_file(y)); year_len.append(T)
    datetimes.append(np.load(cur_file(y), allow_pickle=True)["datetime"])
    if os.path.exists(tp): print(f"  {y}: temp exists"); continue
    mm = npformat.open_memmap(tp, mode="w+", dtype=np.float16, shape=(T,len(DEPTH_IDX),n_keep))
    t0=time.time()
    for r0,r1,a in stream_current(cur_file(y), DEPTH_IDX, keep_cols):
        mm[r0:r1] = np.where(np.isnan(a), 0.0, a).astype(np.float16)
    mm.flush(); del mm
    print(f"  {y}: {T} steps -> temp ({time.time()-t0:.0f}s)")

TimeList = np.concatenate(datetimes); T_total=int(sum(year_len))
print(f"T_total={T_total:,}")

## Pass 2 — evaluate each design, write one file per kite

In [ ]:
def evaluate_design(did):
    d = load_design(did)
    out_path = os.path.join(OUT_DIR, f"{d['name']}.npz")
    print("="*64); print(f"design {did} -> {d['name']}")
    if os.path.exists(out_path): print("  exists, skip"); return

    epu_tmp = os.path.join(TMP_DIR, f"_epu_{did}.npy")
    Energy_pu = npformat.open_memmap(epu_tmp, mode="w+", dtype=np.float16,
                                     shape=(T_total, len(OPERATING_DEPTHS), n_keep))
    off=0; t0=time.time()
    for y,yl in zip(YEARS, year_len):
        cur = np.load(os.path.join(TMP_DIR, f"_curr_{y}.npy"))  # (yl,3,n_keep) f16
        for di in range(len(OPERATING_DEPTHS)):
            vv, row = d["vv"], d["rows"][di]
            sp = cur[:,di,:].astype(np.float32)
            P = np.interp(sp, vv, row)
            P[(sp < vv[0]) | (sp > vv[-1])] = 0.0     # matches fill_value=0.0
            Energy_pu[off:off+yl, di, :] = (P / d["rated"]).astype(np.float16)
        off += yl
    print(f"  power {time.time()-t0:.0f}s")

    CF = np.asarray(Energy_pu, np.float32).mean(axis=0)     # (3, n_keep)
    annual_cost = FCR*d["capex"] + d["opex"]
    annual_energy_mwh = CF * d["rated"] * 8760 / 1000.0
    with np.errstate(divide="ignore", invalid="ignore"):
        LCOE = np.where(annual_energy_mwh > 0, annual_cost/annual_energy_mwh, np.inf)

    np.savez_compressed(out_path,
        Energy_pu=np.asarray(Energy_pu),
        LatLong=LatLong, Depth_m=np.array(OPERATING_DEPTHS),
        AnnualizedCost=np.full(n_keep, (FCR*d["capex"]+d["opex"])/1e6, np.float32),
        RatedPower=np.float64(d["rated"]/1000.0),
        TimeList=TimeList, NumberOfCellsPerSite=np.ones(n_keep),
        ResolutionDegrees=np.float32(RES_DEG), ResolutionKm=np.float32(RES_KM),
        ValidSites=ValidSites, LCOE=LCOE.astype(np.float16),
        CapacityFactor=CF.astype(np.float16), CAPEX=d["capex"], OPEX=d["opex"])
    del Energy_pu; os.remove(epu_tmp)
    finite=np.isfinite(LCOE)&ValidSites
    print(f"  saved {d['name']}.npz  ({n_keep:,} sites x {len(OPERATING_DEPTHS)} depths, "
          f"{os.path.getsize(out_path)/1024**3:.2f} GB) | mean CF {CF[ValidSites].mean():.3f} "
          f"| median LCOE ${np.median(LCOE[finite]):.0f}")

for did in DEVICE_IDS: evaluate_design(did)

# cleanup extracted current temps (comment out to keep for re-runs)
for y in YEARS:
    tp=os.path.join(TMP_DIR,f"_curr_{y}.npy")
    if os.path.exists(tp): os.remove(tp)
print("Done.")